# bigdata7: 机器学习模型流式部署与实时打标 (代码说明版)

FIX_MARKER: UTF8_OK_EXECUTED_2026-04-25_14-36

## 0) Environment, Paths, and Encoding

- Notebook file encoding is UTF-8 (`.ipynb` JSON).
- CSV read/write explicitly uses `encoding='utf-8'`.
- Output directory prefers `G:\\Users\\caoruijie\\big data`; if not writable, auto-fallback to `./task7_outputs`.

In [1]:
import csv
import queue
import threading
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAW_COLUMNS = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]
RNG_SEED = 42
TRAIN_SAMPLE_ROWS = 20_000

DATA_CSV_CANDIDATES = [
    Path(r"C:\Users\caoruijie\Desktop\UserBehavior.csv"),
    Path(r"G:\Users\caoruijie\big data\UserBehavior.csv"),
]
DATA_PARQUET_CANDIDATES = [
    Path(r"G:\Users\caoruijie\big data\m1_final_clean.parquet"),
]
OUTPUT_DIR_CANDIDATES = [
    Path(r"G:\Users\caoruijie\big data"),
    Path.cwd() / "task7_outputs",
]


def first_existing(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def choose_writable_dir(candidates: List[Path]) -> Path:
    for d in candidates:
        try:
            d.mkdir(parents=True, exist_ok=True)
            test_file = d / ".__write_test__"
            test_file.write_text("ok", encoding="utf-8")
            test_file.unlink()
            return d
        except Exception:
            continue
    raise PermissionError("No writable output directory found.")


DATA_CSV = first_existing(DATA_CSV_CANDIDATES)
DATA_PARQUET = first_existing(DATA_PARQUET_CANDIDATES)
OUTPUT_DIR = choose_writable_dir(OUTPUT_DIR_CANDIDATES)

MODEL_PATH = OUTPUT_DIR / "model.pkl"
SCORED_CSV = OUTPUT_DIR / "scored_events.csv"
BENCH_CSV = OUTPUT_DIR / "task7_benchmark.csv"

print("DATA_CSV:", DATA_CSV)
print("DATA_PARQUET:", DATA_PARQUET)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("SCORED_CSV:", SCORED_CSV)
print("BENCH_CSV:", BENCH_CSV)

DATA_CSV: C:\Users\caoruijie\Desktop\UserBehavior.csv
DATA_PARQUET: G:\Users\caoruijie\big data\m1_final_clean.parquet
OUTPUT_DIR: G:\Users\caoruijie\big data
MODEL_PATH: G:\Users\caoruijie\big data\model.pkl
SCORED_CSV: G:\Users\caoruijie\big data\scored_events.csv
BENCH_CSV: G:\Users\caoruijie\big data\task7_benchmark.csv


## 1) 任务1: Offline Training and Model Serialization

Required by assignment:
- label: `behavior_type == 'buy'` as positive class
- pipeline includes imputation + categorical encoding + scaling + classifier
- 5-fold CV metrics: Accuracy and AUC
- save model via `joblib.dump(...)`

In [2]:
def normalize_behavior(v: object) -> str:
    s = str(v).strip().lower()
    if s in {"buy", "purchase"}:
        return "buy"
    if s in {"pv", "view"}:
        return "view"
    if s in {"cart", "fav"}:
        return s
    return s


def load_raw_sample(max_rows: int = 20_000, oversample_factor: int = 3) -> pd.DataFrame:
    target_rows = int(max_rows * max(1, oversample_factor))

    if DATA_CSV is not None:
        df = pd.read_csv(
            DATA_CSV,
            names=RAW_COLUMNS,
            header=None,
            nrows=target_rows,
            encoding="utf-8",
        )
    elif DATA_PARQUET is not None:
        df = (
            pl.scan_parquet(str(DATA_PARQUET))
            .select(RAW_COLUMNS)
            .limit(target_rows)
            .collect()
            .to_pandas()
        )
    else:
        raise FileNotFoundError("No valid source found (CSV/parquet).")

    for col in ["user_id", "item_id", "category_id", "timestamp"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["behavior_type"] = df["behavior_type"].map(normalize_behavior)
    df = df.dropna(subset=["user_id", "item_id", "category_id", "timestamp", "behavior_type"])

    if len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=RNG_SEED)

    return df.reset_index(drop=True)


def build_feature_frame(raw_df: pd.DataFrame, with_label: bool = True):
    df = raw_df.copy()

    for col in ["user_id", "item_id", "category_id", "timestamp"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    ts = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
    hour = ts.dt.hour
    dow = ts.dt.dayofweek

    features = pd.DataFrame(
        {
            "user_id": df["user_id"],
            "item_id": df["item_id"],
            "category_id": df["category_id"],
            "hour": hour,
            "dayofweek": dow,
            "is_weekend": dow.isin([5, 6]).astype(float),
            "hour_bucket": hour.fillna(-1).astype(int).astype(str),
            "day_bucket": dow.fillna(-1).astype(int).astype(str),
        }
    )

    if not with_label:
        return features

    y = df["behavior_type"].map(normalize_behavior).eq("buy").astype(int)
    return features, y


raw_train = load_raw_sample(max_rows=TRAIN_SAMPLE_ROWS, oversample_factor=4)
X_train, y_train = build_feature_frame(raw_train)

numeric_features = ["user_id", "item_id", "category_id", "hour", "dayofweek", "is_weekend"]
categorical_features = ["hour_bucket", "day_bucket"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RNG_SEED)),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
cv_result = cross_validate(
    clf,
    X_train,
    y_train,
    cv=cv,
    scoring={"acc": "accuracy", "auc": "roc_auc"},
    n_jobs=1,
)

print(f"train_rows = {len(X_train):,}")
print(f"positive_ratio = {y_train.mean():.4f}")
print(f"cv_accuracy_mean = {cv_result['test_acc'].mean():.4f}")
print(f"cv_auc_mean = {cv_result['test_auc'].mean():.4f}")

clf.fit(X_train, y_train)
joblib.dump(clf, MODEL_PATH)
print("saved model ->", MODEL_PATH, "size_bytes =", MODEL_PATH.stat().st_size)

train_rows = 20,000
positive_ratio = 0.0207
cv_accuracy_mean = 0.5768
cv_auc_mean = 0.5811
saved model -> G:\Users\caoruijie\big data\model.pkl size_bytes = 4938


In [3]:
# Task1 model asset check
loaded_model = joblib.load(MODEL_PATH)
probe_X = X_train.head(5).copy()
probe_pred = loaded_model.predict(probe_X)
probe_prob = loaded_model.predict_proba(probe_X)[:, 1]

pd.DataFrame({
    "predicted_label": probe_pred.astype(int),
    "buy_probability": probe_prob.astype(float),
})

,predicted_label,buy_probability
0,1,0.517870
1,1,0.514999
2,0,0.265310
3,1,0.630120
4,0,0.351158


### 任务1 简要分析 / Task1 Analysis
- 按任务书要求，已输出 5 折 CV 的 Accuracy 与 AUC。
- `model.pkl` 包含预处理 + 分类器，可直接用于流式在线推理。
- 若 `cv_auc_mean` 明显 > 0.5，则说明模型对“购买 vs 非购买”具备可用区分能力。

## 2) 任务2: Loading Strategy Benchmark

In [4]:
def benchmark_loading_strategies(raw_events: pd.DataFrame, n_events: int = 100) -> pd.DataFrame:
    sample_raw = raw_events.head(n_events).copy()
    sample_X = build_feature_frame(sample_raw, with_label=False)

    t0 = time.perf_counter()
    model_a = joblib.load(MODEL_PATH)
    for i in range(len(sample_X)):
        _ = model_a.predict(sample_X.iloc[[i]])
    elapsed_a = time.perf_counter() - t0

    t1 = time.perf_counter()
    for i in range(len(sample_X)):
        model_b = joblib.load(MODEL_PATH)
        _ = model_b.predict(sample_X.iloc[[i]])
    elapsed_b = time.perf_counter() - t1

    out = pd.DataFrame([
        {
            "scheme": "A_once_load_outside_loop",
            "total_sec": elapsed_a,
            "avg_sec_per_event": elapsed_a / len(sample_X),
            "model_load_count": 1,
        },
        {
            "scheme": "B_reload_inside_loop",
            "total_sec": elapsed_b,
            "avg_sec_per_event": elapsed_b / len(sample_X),
            "model_load_count": len(sample_X),
        },
    ])
    out["slowdown_vs_A"] = out["total_sec"] / elapsed_a
    return out


bench_source = load_raw_sample(max_rows=120, oversample_factor=1)
bench_df = benchmark_loading_strategies(bench_source, n_events=100)
bench_df.to_csv(BENCH_CSV, index=False, encoding="utf-8")
bench_df

,scheme,total_sec,avg_sec_per_event,model_load_count,slowdown_vs_A
0,A_once_load_outside_loop,0.275903,0.002759,1,1.000000
1,B_reload_inside_loop,0.464591,0.004646,100,1.683893


### 任务2 简要分析 / Task2 Analysis
- 方案A（循环外一次加载）是正确工程实践，`model_load_count=1`。
- 方案B会把磁盘 I/O 和反序列化开销放大到每条记录上。
- 对比指标 `slowdown_vs_A` 直接用于量化性能差距。

## 3) 任务3: End-to-End Real-time Scoring

In [5]:
def iter_csv_events(path: Path, limit: Optional[int] = None) -> Iterable[Dict[str, str]]:
    if path is None or (not path.exists()):
        raise FileNotFoundError(path)

    count = 0
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row or len(row) < 5:
                continue
            event = {RAW_COLUMNS[i]: row[i] for i in range(5)}
            yield event
            count += 1
            if limit is not None and count >= limit:
                break


def event_to_feature_df(event: Dict[str, str]) -> pd.DataFrame:
    raw = pd.DataFrame([{k: event.get(k, np.nan) for k in RAW_COLUMNS}])
    return build_feature_frame(raw, with_label=False)


class CSVProducer(threading.Thread):
    def __init__(self, q: queue.Queue, done_event: threading.Event, csv_path: Path, max_events: int, rate_eps: float):
        super().__init__(daemon=True)
        self.q = q
        self.done_event = done_event
        self.csv_path = csv_path
        self.max_events = max_events
        self.rate_eps = max(rate_eps, 0.0)
        self.produced = 0

    def run(self):
        try:
            for event in iter_csv_events(self.csv_path, limit=self.max_events):
                self.q.put(event)
                self.produced += 1
                if self.rate_eps > 0:
                    time.sleep(1.0 / self.rate_eps)
        finally:
            self.done_event.set()


class ScoringConsumer(threading.Thread):
    def __init__(self, q: queue.Queue, done_event: threading.Event, model_path: Path, output_csv: Path, print_every: int = 100):
        super().__init__(daemon=True)
        self.q = q
        self.done_event = done_event
        self.model_path = model_path
        self.output_csv = output_csv
        self.print_every = max(1, print_every)
        self.scored = 0
        self.positive = 0
        self.failed = 0

    def run(self):
        model = joblib.load(self.model_path)
        fieldnames = RAW_COLUMNS + ["predicted_label", "buy_probability", "error"]

        with open(self.output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            while True:
                if self.done_event.is_set() and self.q.empty():
                    break

                try:
                    event = self.q.get(timeout=0.1)
                except queue.Empty:
                    continue

                try:
                    X_one = event_to_feature_df(event)
                    pred = int(model.predict(X_one)[0])
                    prob = float(model.predict_proba(X_one)[0][1])
                    err = ""
                except Exception as e:
                    pred = -1
                    prob = -1.0
                    err = str(e)
                    self.failed += 1

                if pred == 1:
                    self.positive += 1

                out = dict(event)
                out["predicted_label"] = pred
                out["buy_probability"] = prob
                out["error"] = err
                writer.writerow(out)

                self.scored += 1
                if self.scored % self.print_every == 0:
                    print(f"scored={self.scored}, positive={self.positive}, failed={self.failed}, latest_prob={prob:.4f}")

                self.q.task_done()


def run_task3_pipeline(max_events: int = 500, rate_eps: float = 300, queue_size: int = 300, print_every: int = 100):
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"model not found: {MODEL_PATH}. Run task1 first.")
    if DATA_CSV is None:
        raise FileNotFoundError("CSV source not found.")

    q = queue.Queue(maxsize=queue_size)
    done = threading.Event()

    producer = CSVProducer(q=q, done_event=done, csv_path=DATA_CSV, max_events=max_events, rate_eps=rate_eps)
    consumer = ScoringConsumer(q=q, done_event=done, model_path=MODEL_PATH, output_csv=SCORED_CSV, print_every=print_every)

    t0 = time.perf_counter()
    producer.start()
    consumer.start()
    producer.join()
    consumer.join()
    elapsed = time.perf_counter() - t0

    scored_df = pd.read_csv(SCORED_CSV, encoding="utf-8")
    valid_probs = scored_df.loc[scored_df["buy_probability"] >= 0, "buy_probability"]

    summary = pd.DataFrame([
        {
            "produced": producer.produced,
            "scored": consumer.scored,
            "failed": consumer.failed,
            "positive": consumer.positive,
            "positive_ratio": (consumer.positive / consumer.scored) if consumer.scored else 0.0,
            "avg_buy_probability": valid_probs.mean() if len(valid_probs) else np.nan,
            "elapsed_sec": elapsed,
            "throughput_eps": (consumer.scored / elapsed) if elapsed > 0 else np.nan,
        }
    ])
    return summary, scored_df.head(10)


task3_summary, task3_preview = run_task3_pipeline(max_events=500, rate_eps=300, queue_size=300, print_every=100)
print("scored csv ->", SCORED_CSV)
task3_summary

scored=100, positive=60, failed=0, latest_prob=0.6214


scored=200, positive=120, failed=0, latest_prob=0.5241


scored=300, positive=172, failed=0, latest_prob=0.5677


scored=400, positive=194, failed=0, latest_prob=0.5924


scored=500, positive=256, failed=0, latest_prob=0.3526
scored csv -> G:\Users\caoruijie\big data\scored_events.csv


,produced,scored,failed,positive,positive_ratio,avg_buy_probability,elapsed_sec,throughput_eps
0,500,500,0,256,0.512,0.503833,3.884789,128.707132


In [6]:
task3_preview

,user_id,item_id,category_id,behavior_type,timestamp,predicted_label,buy_probability,error
0,1,2268318,2520377,pv,1511544070,1,0.640463,NaN
1,1,2333346,2520771,pv,1511561733,0,0.427467,NaN
2,1,2576651,149192,pv,1511572885,1,0.649830,NaN
3,1,3830808,4181361,pv,1511593493,0,0.261878,NaN
4,1,4365585,2520377,pv,1511596146,0,0.306356,NaN
5,1,4606018,2735466,pv,1511616481,0,0.406595,NaN
6,1,230380,411153,pv,1511644942,0,0.290626,NaN
7,1,3827899,2920476,pv,1511713473,1,0.562663,NaN
8,1,3745169,2891509,pv,1511725471,1,0.690674,NaN
9,1,1531036,2920476,pv,1511733732,0,0.475252,NaN


### 任务3 简要分析 / Task3 Analysis
- 打标结果落盘到 `scored_events.csv`，字段包含原始列 + `predicted_label` + `buy_probability` + `error`。
- 关键容错策略：单条记录失败时标记 `predicted_label=-1`，不中断消费线程。
- 可直接读取概要表中的 `positive_ratio` 与 `avg_buy_probability` 作为结果统计。

## 4) 任务4: Micro-Batch Throughput Optimization

In [7]:
def run_single_inference_benchmark(events: List[Dict[str, str]]) -> Dict[str, float]:
    model = joblib.load(MODEL_PATH)
    processed = 0
    failed = 0

    t0 = time.perf_counter()
    for event in events:
        try:
            X_one = event_to_feature_df(event)
            _ = model.predict(X_one)
            _ = model.predict_proba(X_one)
        except Exception:
            failed += 1
        processed += 1
    elapsed = time.perf_counter() - t0

    return {
        "mode": "single_B1",
        "processed": processed,
        "failed": failed,
        "elapsed_sec": elapsed,
        "throughput_eps": processed / elapsed if elapsed > 0 else np.nan,
    }


def run_micro_batch_benchmark(events: List[Dict[str, str]], batch_size: int = 50, batch_timeout: float = 0.5) -> Dict[str, float]:
    model = joblib.load(MODEL_PATH)
    q = queue.Queue()
    for e in events:
        q.put(e)

    processed = 0
    failed = 0
    buffer: List[Dict[str, str]] = []
    last_flush = time.perf_counter()

    def flush_buffer():
        nonlocal processed, failed, buffer, last_flush
        if not buffer:
            return
        try:
            X_batch = pd.concat([event_to_feature_df(e) for e in buffer], ignore_index=True)
            _ = model.predict(X_batch)
            _ = model.predict_proba(X_batch)
        except Exception:
            failed += len(buffer)
        processed += len(buffer)
        buffer = []
        last_flush = time.perf_counter()

    t0 = time.perf_counter()
    while not q.empty() or buffer:
        try:
            event = q.get(timeout=0.1)
            buffer.append(event)
            q.task_done()
        except queue.Empty:
            pass

        if len(buffer) >= batch_size or (buffer and (time.perf_counter() - last_flush > batch_timeout)):
            flush_buffer()

    flush_buffer()
    elapsed = time.perf_counter() - t0

    return {
        "mode": f"micro_batch_B{batch_size}",
        "processed": processed,
        "failed": failed,
        "elapsed_sec": elapsed,
        "throughput_eps": processed / elapsed if elapsed > 0 else np.nan,
    }


if DATA_CSV is None:
    raise FileNotFoundError("CSV source not found for task4 benchmark.")

events_1000 = list(iter_csv_events(DATA_CSV, limit=1000))
res_single = run_single_inference_benchmark(events_1000)
res_batch = run_micro_batch_benchmark(events_1000, batch_size=50, batch_timeout=0.5)

throughput_df = pd.DataFrame([res_single, res_batch])
throughput_df["speedup_vs_single"] = throughput_df["throughput_eps"] / res_single["throughput_eps"]
throughput_df

,mode,processed,failed,elapsed_sec,throughput_eps,speedup_vs_single
0,single_B1,1000,0,6.630870,150.809773,1.000000
1,micro_batch_B50,1000,0,1.319039,758.127870,5.027047


### 任务4 简要分析 / Task4 Analysis
- Micro-Batch 用 `batch_size` 提升吞吐量，用 `batch_timeout` 避免低流量长时间不出结果。
- 若只保留批量触发（去掉 timeout），低速流量下会出现“积压但不出标签”问题。

## 运行顺序建议 / Recommended Order
1. 先跑 Task1 生成 `model.pkl`。
2. 再跑 Task2 产生加载策略对比表（`task7_benchmark.csv`）。
3. 执行 Task3 生成 `scored_events.csv` 及前10行预览。
4. 最后跑 Task4 对比逐条推理与 Micro-Batch 吞吐。